# DDPM Model

In [1]:

import os
import random
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image, ImageFile
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchvision.utils import make_grid, save_image
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
from sklearn.metrics import mean_squared_error

# Enable partial loading
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Dataset paths
train_path = "/content/drive/MyDrive/processed/train"
test_path = "/content/drive/MyDrive/processed/test"
CLASSES = ['inside', 'outside', 'food', 'drink']
NUM_CLASSES = len(CLASSES)

TRAIN_SAMPLES_PER_CLASS = 200  # reduced
TEST_SAMPLES_PER_CLASS = 50    # reduced

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

class RobustYelpDataset(Dataset):
    def __init__(self, root_dir, transform=None, samples_per_class=10):
        self.image_paths = []
        self.labels = []
        self.transform = transform

        for label_idx, cls in enumerate(CLASSES):
            cls_path = os.path.join(root_dir, cls)
            if not os.path.exists(cls_path):
                print(f"Missing: {cls_path}")
                continue

            all_images = [f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))]
            sampled = random.sample(all_images, min(samples_per_class, len(all_images)))
            for img_name in sampled:
                self.image_paths.append(os.path.join(cls_path, img_name))
                self.labels.append(label_idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        try:
            img = Image.open(path).convert('RGB')
        except:
            img = Image.new('RGB', (64, 64), (0, 0, 0))
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

train_dataset = RobustYelpDataset(train_path, transform, TRAIN_SAMPLES_PER_CLASS)
test_dataset = RobustYelpDataset(test_path, transform, TEST_SAMPLES_PER_CLASS)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)  # reduced batch size
test_loader = DataLoader(test_dataset, batch_size=16)

# === MODEL ===
class DummyUNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Conv2d(3, 32, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(32, 32, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(32, 3, 3, padding=1)
        )

    def forward(self, x, t):
        return self.net(x)

class DDPM(torch.nn.Module):
    def __init__(self, model, timesteps=100, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        self.model = model
        self.timesteps = timesteps
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.betas = torch.linspace(beta_start, beta_end, timesteps).to(self.device)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def q_sample(self, x_start, t, noise):
        sqrt_ab = torch.sqrt(self.alpha_bars[t])[:, None, None, None]
        sqrt_one_minus_ab = torch.sqrt(1 - self.alpha_bars[t])[:, None, None, None]
        return sqrt_ab * x_start + sqrt_one_minus_ab * noise

    def forward(self, x, t):
        noise = torch.randn_like(x)
        x_t = self.q_sample(x, t, noise)
        return self.model(x_t, t), noise

    def denoise(self, noisy, t):
        pred_noise = self.model(noisy, t)
        alpha = self.alphas[t].view(-1, 1, 1, 1)
        alpha_bar = self.alpha_bars[t].view(-1, 1, 1, 1)
        return (noisy - (1 - alpha) / torch.sqrt(1 - alpha_bar) * pred_noise) / torch.sqrt(alpha)

    def sample(self, n=8):  # reduced n
        self.eval()
        x = torch.randn(n, 3, 64, 64).to(self.device)
        for t in reversed(range(self.timesteps)):
            t_tensor = torch.full((n,), t, dtype=torch.long, device=self.device)
            beta = self.betas[t]
            alpha = self.alphas[t]
            alpha_bar = self.alpha_bars[t]
            pred_noise = self.model(x, t_tensor)
            noise = torch.randn_like(x) if t > 0 else 0
            x = (1 / torch.sqrt(alpha)) * (x - (1 - alpha) / torch.sqrt(1 - alpha_bar) * pred_noise) + torch.sqrt(beta) * noise
        return torch.clamp(x, -1, 1)

# === TRAINING ===
def train_ddpm(ddpm_model, dataloader, epochs=10, lr=1e-4):  # reduced epochs and lr
    optimizer = torch.optim.Adam(ddpm_model.parameters(), lr=lr)
    loss_fn = torch.nn.MSELoss()
    ddpm_model.train()
    ddpm_model.to(ddpm_model.device)

    for epoch in range(epochs):
        total_loss = 0
        for images, _ in dataloader:
            images = images.to(ddpm_model.device)
            t = torch.randint(0, ddpm_model.timesteps, (images.size(0),), device=ddpm_model.device)
            pred_noise, true_noise = ddpm_model(images, t)
            loss = loss_fn(pred_noise, true_noise)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} | Loss: {total_loss / len(dataloader):.4f}")

# === FID + IS + IMAGE GENERATION ===
def evaluate_generated_images(ddpm_model, real_images_loader):
    ddpm_model.eval()
    print("Generating samples for evaluation...")
    samples = ddpm_model.sample(n=8).detach().cpu()
    samples = (samples + 1) / 2.0  # scale to [0,1]

    real_images = []
    for imgs, _ in real_images_loader:
        real_images.append(imgs[:len(samples)])
        break
    real_images = torch.cat(real_images, dim=0)
    real_images = (real_images + 1) / 2.0  # scale to [0,1]

    # Convert to uint8 [0,255]
    samples_uint8 = (samples * 255).clamp(0, 255).to(torch.uint8)
    real_images_uint8 = (real_images * 255).clamp(0, 255).to(torch.uint8)

    fid = FrechetInceptionDistance(feature=64).to(ddpm_model.device)
    fid.update(samples_uint8.to(ddpm_model.device), real=False)
    fid.update(real_images_uint8.to(ddpm_model.device), real=True)
    fid_score = fid.compute().item()

    is_metric = InceptionScore().to(ddpm_model.device)
    is_metric.update(samples_uint8.to(ddpm_model.device))
    is_score = is_metric.compute()

    print(f"FID Score: {fid_score:.4f}, IS Score: {is_score[0]:.4f} (std: {is_score[1]:.4f})")
    save_image(samples, "generated_ddpm_samples.png", nrow=4)
    return fid_score, is_score

# === EXECUTE ===
unet = DummyUNet()
ddpm = DDPM(unet)
train_ddpm(ddpm, train_loader, epochs=10)
fid_score, is_score = evaluate_generated_images(ddpm, test_loader)
print(f"\u2705 FID: {fid_score:.2f}, IS: {is_score[0]:.2f} ± {is_score[1]:.2f}")
# plot_original_denoised_noisy_grid(ddpm, test_dataset, noise_level=0.3, count=6)


Epoch 1 | Loss: 0.9598
Epoch 2 | Loss: 0.8572
Epoch 3 | Loss: 0.7255
Epoch 4 | Loss: 0.6100
Epoch 5 | Loss: 0.4857
Epoch 6 | Loss: 0.3847
Epoch 7 | Loss: 0.3271
Epoch 8 | Loss: 0.3223
Epoch 9 | Loss: 0.2990
Epoch 10 | Loss: 0.2809
Generating samples for evaluation...


/usr/local/lib/python3.11/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


FID Score: 75.6729, IS Score: 1.0000 (std: 0.0000)
✅ FID: 75.67, IS: 1.00 ± 0.00
